1. How do the number of crashes vary by hour, month and year in each borough?
   Measure: crash counts grouped by hour of day, month and year, split by borough.

2. Which borough has the highest average number of injuries per crash?
   Measure: total persons injured / total crashes, per borough. The borough with the
   highest average ranks first.

3. Which contributing factors are most common overall, and which are
   over-represented in crashes with injuries?
   Measure: each factor's share of all crashes vs its share of crashes with at least one
   injury, compared by rank.

4. How do injuries and deaths differ among pedestrians, cyclists and motorists?
   Measure: injured and killed counts per group, with killed per injured as a severity
   rate, broken down by borough, hour and contributing factor.

5. Which vehicle types appear most often in crashes in each borough?
   Measure: for each borough, rank vehicle types by the number of crashes involving them.

How has the average number of injuries per crash changed from year to year?
   Measure: injuries per crash for each full year, and the year-over-year percentage change
   = (this year - last year) / last year * 100.

In [19]:
import pandas as pd
import numpy as np
import matplotlib as mpl
import seaborn as sns
import jupyterlab as jl

df = pd.read_csv('Motor_Vehicle_Collisions_-_Crashes.csv', low_memory=False )

print(df.shape)
df.info()


(449586, 29)
<class 'pandas.DataFrame'>
RangeIndex: 449586 entries, 0 to 449585
Data columns (total 29 columns):
 #   Column                         Non-Null Count   Dtype  
---  ------                         --------------   -----  
 0   CRASH DATE                     449586 non-null  str    
 1   CRASH TIME                     449586 non-null  str    
 2   BOROUGH                        348242 non-null  str    
 3   ZIP CODE                       322555 non-null  float64
 4   LATITUDE                       425039 non-null  float64
 5   LONGITUDE                      425039 non-null  float64
 6   LOCATION                       425039 non-null  str    
 7   ON STREET NAME                 319059 non-null  str    
 8   CROSS STREET NAME              130523 non-null  str    
 9   OFF STREET NAME                232292 non-null  str    
 10  NUMBER OF PERSONS INJURED      449586 non-null  int64  
 11  NUMBER OF PERSONS KILLED       449586 non-null  int64  
 12  NUMBER OF PEDESTRIANS INJURE

# Data Issues and Cleaning Log

**Dataset:** NYC Motor Vehicle Collisions - Crashes (NYC Open Data)
**Size as downloaded:** 449,586 rows, 29 columns. Each row is one crash.

---

## 2. Duplicates and consistency

**4. Possible duplicate collision IDs**
- **Decision:** Remove only if the rows are identical. If the same ID has different data, investigate first.
- **Why:** Avoid deleting real records.

**5. Injury and death totals may not match their parts**
- **Decision:** Check that persons injured and persons killed equal the sum of the pedestrian, cyclist and motorist columns. If they differ, decide which columns to trust and document it.
- **Why:** The totals are used in Q2 and Q4.

---

## 3. Missing and invalid values

**6. Borough is missing in many rows (about 22.5%)**
- **Decision:** Decide after checking. Options: label as "Unknown" for borough analyses, recover from latitude and longitude, or exclude from those analyses only.
- **Why:** Q1, Q2 and Q5 depend on borough. These crashes still count for the other questions.

**8. Latitude and longitude may be 0 or outside New York City**
- **Decision:** Check for 0 and for values outside the NYC range. Set invalid values to missing.
- **Why:** They would plot far outside the city.

**9. Some street names are only numbers, and some may be placeholders such as 0**
- **Decision:** Check with `value_counts()`. Low priority.
- **Why:** None of the 6 questions use street names. Mention it in the limitations.

**10. Contributing factors include "Unspecified"**
- **Decision:** Decide after checking. Either exclude it from the Q3 ranking and report its share, or include it.
- **Why:** It is not an actual cause, and it may dominate the ranking.

---

## 4. Text standardization

**11. Vehicle type labels differ by case and spelling**
- **Example:** sedan, SEDAN, sedan 2, sedan 4.
- **Decision:** Lowercase, trim spaces, and group into a short list of categories. Keep the raw column and document the grouping rules.
- **Why:** Needed for the vehicle ranking in Q5.

**12. Other text columns may have inconsistent case or extra spaces**
- **Decision:** Apply the same case and spacing checks to borough and contributing factors.
- **Why:** Prevents one category from appearing under several names.

---

## 5. Columns to drop and to create

**13. Location repeats latitude and longitude**
- **Decision:** Confirm it is missing in exactly the same rows as latitude and longitude, then drop it.
- **Why:** It is redundant.

**14. No column marks crashes with injuries**
- **Decision:** Create `injury_crash` (True if at least one person was injured).
- **Why:** Needed for Q3.

**1. Crash date and time are stored as text**
- **Decision:** Convert to datetime and combine into one column. Check the result with `min()` and `max()`. Extract hour, month and year into their own columns.
- **Why:** Needed for filtering, sorting and the time analysis (Q1, Q6).

In [24]:
df_raw = df.copy()  # Keep a copy of the raw data for reference

df['CRASH INFO'] = pd.to_datetime(df['CRASH DATE'] + ' ' + df['CRASH TIME'], format="%m/%d/%Y %H:%M", errors='coerce')  # Combine date and time into a single datetime column

df['CRASH YEAR'] = df['CRASH INFO'].dt.year
df['CRASH MONTH'] = df['CRASH INFO'].dt.month
df['CRASH HOUR'] = df['CRASH INFO'].dt.hour

df['CRASH INFO'].isna().sum()  # Check for any rows where the datetime conversion failed

np.int64(0)

**2. ZIP code is stored as a decimal number**
- **Decision:** Convert to text.
- **Why:** A ZIP code is an identifier, not a quantity.

**7. ZIP codes may be 0 or outside New York City**
- **Decision:** Check for 0 and for values outside the NYC range. Set invalid values to missing.
- **Why:** 0 is a placeholder, not a real ZIP.

In [46]:
df["ZIP CODE"] = df["ZIP CODE"].astype("Int64").astype("string")
df["ZIP CODE"].str.len().value_counts()

zip_num = pd.to_numeric(df["ZIP CODE"], errors="coerce") 

in_range = zip_num.between(10000, 11698)
missing  = zip_num.isna()

outside_range = ~in_range & ~missing

(df['ZIP CODE'] == "0").sum()  # Count how many ZIP codes are "0"

df.loc[outside_range, 'ZIP CODE'] = pd.NA  # Set ZIP codes outside the valid range to missing






**3. Collision ID is stored as text**
- **Decision:** Check that it is unique per crash.
- **Why:** Duplicate crashes would inflate every count.

In [48]:
df["COLLISION_ID"].is_unique
df.duplicated(subset=["COLLISION_ID"]).sum()  # Count how many duplicate COLLISION_IDs there are

np.int64(0)